## Demo of Variational Autoencoders

This is a basic demo of variational autoencoders on MNIST data using **PyTorch**, adapted from the post [Variational AutoEncoders and Image Generation with Keras](https://dropsofai.com/variational-autoencoders-and-image-generation-with-keras/)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Load data and display a few images

In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainX = train_dataset.data.numpy()
trainy = train_dataset.targets.numpy()
testX = test_dataset.data.numpy()
testy = test_dataset.targets.numpy()

print('Training data shape: X=%s, y=%s' % (trainX.shape, trainy.shape))
print('Testing data shape: X=%s, y=%s' % (testX.shape, testy.shape))

ind = np.random.choice(np.arange(len(trainX)), size=5)
for i in ind:
    plt.figure(figsize=(2, 2))
    plt.imshow(trainX[i], cmap='gray')
    plt.axis('off')
    plt.title('')
plt.show()

### Prepare DataLoaders and normalized arrays

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Normalized numpy arrays for visualization
train_data = trainX.astype('float32') / 255  # (60000, 28, 28)
test_data = testX.astype('float32') / 255    # (10000, 28, 28)

# Building the VAE

### Encoder network

This is the network that outputs the variational mean and log-variance. It has three convolutional layers and a dense layer.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5),   # (N, 64, 24, 24)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # (N, 64, 12, 12)
            nn.Conv2d(64, 64, kernel_size=3),  # (N, 64, 10, 10)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # (N, 64, 5, 5)
            nn.Conv2d(64, 32, kernel_size=3),  # (N, 32, 3, 3)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # (N, 32, 1, 1)
        )
        self.fc = nn.Linear(32, 16)
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_log_var = nn.Linear(16, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)      # flatten -> (N, 32)
        x = self.fc(x)                  # (N, 16)
        mu = self.fc_mu(x)              # (N, latent_dim)
        log_var = self.fc_log_var(x)    # (N, latent_dim)
        return mu, log_var

### Sampling from the variational distribution

This applies the "reparameterization trick" by sampling a standard Gaussian, scaling by the noise level and adding the mean:

$$ Z  = \mu(x) + \sigma(x) \epsilon$$

Since the network outputs the log of the variance, we exponentiate it:

$$ \sigma(x) = \exp\left(\frac{1}{2} \log \sigma^2(x)\right)$$

## Generative network (Decoder)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=3),  # (N, 64, 3, 3)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 64, kernel_size=3),  # (N, 64, 5, 5)
            nn.ReLU(),
            nn.Upsample(scale_factor=2),                # (N, 64, 10, 10)
            nn.ConvTranspose2d(64, 64, kernel_size=3),  # (N, 64, 12, 12)
            nn.ReLU(),
            nn.Upsample(scale_factor=2),                # (N, 64, 24, 24)
            nn.ConvTranspose2d(64, 1, kernel_size=5),   # (N, 1, 28, 28)
            nn.ReLU(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 64, 1, 1)
        x = self.deconv(x)
        return x

## Combining

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        x_recon = self.decoder(z)
        return x_recon, mu, log_var

In [ ]:
model = VAE(latent_dim=2).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters: {total_params:,}')

### Loss Function (Reconstruction Loss + KL-loss)

As not mentioned in class, when generating data, we can add noise. Thus the model is
$ Z \sim N(0,I)$ and then $X | z = N(G(z), \gamma^2 I)$ where the noise level is $\gamma$.
In this case we add a mean-squared error term to the loss function. And MSE minus ELBO is MSE + KL(q, p).

In [ ]:
def vae_loss(x_recon, x, mu, log_var):
    # Reconstruction loss (MSE scaled by image size)
    recon_loss = F.mse_loss(x_recon, x, reduction='mean') * 28 * 28
    # KL divergence: -0.5 * mean(sum(1 + log_var - mu^2 - exp(log_var)))
    kl_loss = -0.5 * torch.mean(
        torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1)
    )
    return recon_loss + kl_loss

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters())

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for data, _ in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        x_recon, mu, log_var = model(data)
        loss = vae_loss(x_recon, data, mu, log_var)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            x_recon, mu, log_var = model(data)
            val_loss += vae_loss(x_recon, data, mu, log_var).item()

    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(test_loader)
    print(f'Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}')

In [ ]:
offset = 400
model.eval()

print('Real Test Images')
for i in range(9):
    plt.subplot(330 + 1 + i)
    plt.imshow(test_data[i + offset], cmap='gray')
    plt.axis('off')
plt.show()

print('Reconstructed Images with Variational Autoencoder')
with torch.no_grad():
    for i in range(9):
        plt.subplot(330 + 1 + i)
        sample = torch.FloatTensor(test_data[i + offset]).unsqueeze(0).unsqueeze(0).to(device)
        output, _, _ = model(sample)
        op_image = output[0, 0].cpu().numpy() * 255
        plt.imshow(op_image, cmap='gray')
        plt.axis('off')
plt.show()

## Latent feature clusters

In [ ]:
model.eval()
x_coords = []
y_coords = []
labels = []

test_tensor = torch.FloatTensor(test_data).unsqueeze(1).to(device)  # (10000, 1, 28, 28)

with torch.no_grad():
    for i in range(10000):
        labels.append(testy[i])
        mu, _ = model.encoder(test_tensor[i].unsqueeze(0))
        x_coords.append(mu[0, 0].item())
        y_coords.append(mu[0, 1].item())

In [ ]:
df = pd.DataFrame()
df['x'] = x_coords
df['y'] = y_coords
df['z'] = ['digit-' + str(k) for k in labels]
labels_arr = np.array(labels)
plt.figure(figsize=(8, 6))
for c in np.unique(labels_arr):
    idx = np.where(labels_arr == c)
    plt.scatter(np.array(x_coords)[idx], np.array(y_coords)[idx], label=c)
plt.legend()
plt.show()

## Image Generation

In [ ]:
x_values = np.linspace(-3, 3, 30)
y_values = np.linspace(-3, 3, 30)

In [ ]:
figure = np.zeros((28 * 30, 28 * 30))
model.eval()
with torch.no_grad():
    for ix, xv in enumerate(x_values):
        for iy, yv in enumerate(y_values):
            latent_point = torch.FloatTensor([[xv, yv]]).to(device)
            generated_image = model.decoder(latent_point)
            figure[ix*28:(ix+1)*28, iy*28:(iy+1)*28] = generated_image[0, 0].cpu().numpy()

plt.figure(figsize=(15, 15))
plt.imshow(figure, cmap='gray', extent=[3, -3, 3, -3])
plt.axis('off')
plt.show()